# Problema del Peluquero Dormido — Análisis Formal y Verificación Experimental

**Universidad del Valle — Departamento de Ingeniería de Sistemas**  
**Sistemas Operativos — Taller Final**  

---

## Frente 1 — Reproducción Determinista del Fallo (*Lost Wakeup*)

### 1. Objetivo
El objetivo de esta sección es reproducir e ilustrar de manera completamente determinista el fallo de concurrencia conocido como **Lost Wakeup** (Despertar Perdido) en la versión incorrecta del problema del peluquero dormido, donde no se emplea una sincronización atómica para la secuencia de verificación de estado y bloqueo (dormir).

## 2. Fundamento Teórico

El fenómeno del **Lost Wakeup** (OSTEP, Capítulo 30) ocurre cuando un hilo lector/receptor se prepara para bloquearse esperando una señal, pero justo antes de efectuar la llamada de bloqueo (p. ej., `wait()`), es interrumpido por el planificador (scheduler). Otro hilo (el emisor) se ejecuta, altera la condición de espera y envía la señal de despertar (`signal` o `set`). Como el primer hilo aún no está técnicamente en estado de espera, la señal se pierde. Cuando el primer hilo reanuda su ejecución, procede a bloquearse y queda suspendido indefinidamente porque la señal ya fue emitida y no volverá a ocurrir.

En el problema del peluquero dormido:
1. El barbero comprueba `waiting == 0` (no hay clientes).
2. Justo antes de irse a dormir (ejecutar `wait()`), llega un cliente.
3. El cliente ve que el barbero está "despierto" (pues el estado aún no es "durmiendo"), por lo que no le envía la señal de despertar (`set()`), limitándose a sentarse en la silla de espera.
4. El barbero reanuda su ejecución, asume que no hay clientes porque ya hizo la evaluación previa, y se duerme (`wait()`).
5. El barbero queda durmiendo indefinidamente (Lost Wakeup) y el cliente esperando en la silla para siempre.

In [19]:
# ===========================================================================
# FRENTE 1 — Implementación INCORRECTA del Problema del Peluquero Dormido
# Propósito: Demostrar el Lost Wakeup de forma determinista
# ===========================================================================

import threading
import time
import random

# Parámetros de la barbería
NUM_CHAIRS = 3          # Número de sillas de espera
INJECT_DELAY = 0.15     # Retardo inyectado para simular la interrupción del scheduler

# Variables de estado compartidas sin protección adecuada de exclusión mutua
waiting = 0             # Clientes esperando
barber_sleeping = False # Estado del barbero
barber_event = threading.Event()  # Evento para despertar al barbero

# Registro de eventos
start_time = time.perf_counter()
event_log = []
log_lock = threading.Lock()  # Exclusión mutua únicamente para el orden del log impreso

def log(actor, message):
    """Registra un evento con marca de tiempo precisa."""
    with log_lock:
        ts = time.perf_counter() - start_time
        entry = f"[T={ts:.4f}s] [{actor}] {message}"
        event_log.append(entry)
        print(entry)

def barber_incorrect():
    global waiting, barber_sleeping
    
    log("BARBERO", "Inicio de turno. Verificando si hay clientes...")
    
    # Verificación del estado de la sala
    # Entre esta lectura de 'waiting' y el bloqueo físico en 'wait()'
    # ocurre la interrupción del scheduler (ventana de vulnerabilidad).
    if waiting == 0:
        log("BARBERO", "No hay clientes. Me preparo para dormir...")
        
        # INYECCIÓN DEL RETARDO:
        # Forzamos al scheduler a dar paso a otro hilo (el cliente) antes de bloquearnos.
        time.sleep(INJECT_DELAY)
        
        # Transición al estado durmiendo y bloqueo
        barber_sleeping = True
        log("BARBERO", "Entrando en estado durmiendo. Ejecutando wait()...")
        
        # Usamos un timeout de 2.0 segundos para el experimento
        woke_up = barber_event.wait(timeout=2.0)
        
        if woke_up:
            barber_sleeping = False
            log("BARBERO", "Desperté. Atendiendo al cliente.")
        else:
            log("BARBERO", " FALLO: Nadie me despertó. Lost Wakeup confirmado.")
    else:
        log("BARBERO", f"Hay {waiting} cliente(s) esperando. Atendiendo...")

def client_incorrect(client_id, arrival_delay):
    global waiting, barber_sleeping
    
    # Simula el viaje del cliente a la barbería
    time.sleep(arrival_delay)
    log(f"CLIENTE-{client_id}", "Llegando a la barbería...")
    
    # Condicion para evaluar sillas libres
    if waiting < NUM_CHAIRS:
        waiting += 1
        log(f"CLIENTE-{client_id}", f"Me senté en una silla. Esperando = {waiting}")
        
        # El cliente comprueba si el barbero duerme
        # Como el barbero está en su ventana de retardo, barber_sleeping es aún False.
        # El cliente asume erróneamente que el barbero está despierto y NO envía la señal.
        if barber_sleeping:
            log(f"CLIENTE-{client_id}", "El barbero está durmiendo. Enviando señal de despertar (set)... ")
            barber_event.set()
        else:
            log(f"CLIENTE-{client_id}", "Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]")
    else:
        log(f"CLIENTE-{client_id}", "Sala llena. Me retiro.")

### 3. Ejecución Experimental del Fallo

In [20]:
print("=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===\n")

# Limpieza e inicialización
event_log.clear()
barber_event.clear()
waiting = 0
barber_sleeping = False

# Hilos: El barbero inicia de inmediato (T=0). El cliente llega en T=0.05s, 
# cayendo exactamente en la ventana de vulnerabilidad del barbero (0s a 0.15s).
t_barber = threading.Thread(target=barber_incorrect)
t_client = threading.Thread(target=client_incorrect, args=(1, 0.05))

t_start = time.perf_counter()
t_barber.start()
t_client.start()

t_barber.join()
t_client.join()
t_duration = time.perf_counter() - t_start

print(f"\nDuración de la simulación: {t_duration:.4f} segundos")
if t_duration >= 2.0:
    print("\nRESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).")
else:
    print("\nRESULTADO: El barbero se despertó a tiempo.")

=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===

[T=0.0060s] [BARBERO] Inicio de turno. Verificando si hay clientes...
[T=0.0061s] [BARBERO] No hay clientes. Me preparo para dormir...
[T=0.0604s] [CLIENTE-1] Llegando a la barbería...
[T=0.0605s] [CLIENTE-1] Me senté en una silla. Esperando = 1
[T=0.0605s] [CLIENTE-1] Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]
[T=0.1604s] [BARBERO] Entrando en estado durmiendo. Ejecutando wait()...
[T=2.1656s] [BARBERO]  FALLO: Nadie me despertó. Lost Wakeup confirmado.

Duración de la simulación: 2.1620 segundos

RESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).


### 4. Análisis de la Intercalación Registrada
A continuación, podemos examinar la bitácora exacta de eventos para rastrear el entrelazamiento temporal (*interleaving*) que causó la pérdida de la señal de despertar:

In [21]:
print("=== SECUENCIA DE INTERCALACIÓN REGISTRADA ===\n")
for entry in event_log:
    print(entry)

print("\nExplicación del entrelazamiento:")
print("1. El barbero inicia y verifica 'waiting == 0', lo cual es verdadero.")
print("2. El planificador suspende al barbero (simulado por INJECT_DELAY).")
print("3. El cliente llega, incrementa 'waiting' a 1, e inspecciona 'barber_sleeping'.")
print("4. Dado que el barbero aún no ha ejecutado 'wait()', 'barber_sleeping' es False.")
print("5. El cliente asume que el barbero lo atenderá y finaliza sin enviar la señal (set).")
print("6. El barbero reanuda, establece 'barber_sleeping = True' y se bloquea en 'wait()'.")
print("7. El barbero queda suspendido hasta que expira el timeout, demostrando el Lost Wakeup.")

=== SECUENCIA DE INTERCALACIÓN REGISTRADA ===

[T=0.0060s] [BARBERO] Inicio de turno. Verificando si hay clientes...
[T=0.0061s] [BARBERO] No hay clientes. Me preparo para dormir...
[T=0.0604s] [CLIENTE-1] Llegando a la barbería...
[T=0.0605s] [CLIENTE-1] Me senté en una silla. Esperando = 1
[T=0.0605s] [CLIENTE-1] Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]
[T=0.1604s] [BARBERO] Entrando en estado durmiendo. Ejecutando wait()...
[T=2.1656s] [BARBERO]  FALLO: Nadie me despertó. Lost Wakeup confirmado.

Explicación del entrelazamiento:
1. El barbero inicia y verifica 'waiting == 0', lo cual es verdadero.
2. El planificador suspende al barbero (simulado por INJECT_DELAY).
3. El cliente llega, incrementa 'waiting' a 1, e inspecciona 'barber_sleeping'.
4. Dado que el barbero aún no ha ejecutado 'wait()', 'barber_sleeping' es False.
5. El cliente asume que el barbero lo atenderá y finaliza sin enviar la señal (set).
6. El barbero reanuda, establece 'barber_sleeping = True' y s

---

## Frente 2 — Modelo e Invariantes del Sistema

### 1. Modelo Formal del Sistema

Para analizar formalmente la concurrencia en la barbería, definimos los componentes del sistema:

#### 1.1 Actores y sus Estados

1. **Barbero (Hilo Único):**
   - `DURMIENDO`: El barbero está inactivo en su silla esperando que llegue un cliente.
   - `VERIFICANDO`: El barbero examina si hay clientes en la sala de espera.
   - `TRABAJANDO`: El barbero está realizando un corte de cabello.

2. **Clientes (N Hilos Concurrentes):**
   - `VIAJANDO`: Simula la llegada del cliente con tiempos aleatorios.
   - `LLEGANDO`: El cliente entra a la barbería y evalúa la disponibilidad de sillas.
   - `ESPERANDO`: El cliente está sentado en la sala de espera.
   - `RECIBIENDO_CORTE`: El cliente está en la silla del barbero mientras este le corta el cabello.
   - `ATENDIDO`: Estado final de éxito; el cliente sale de la barbería.
   - `RECHAZADO`: Estado final de descarte (sala llena); el cliente se retira de inmediato.

#### 1.2 Recursos Compartidos y Variables

- **Sala de Espera:** Capacidad limitada de $N$ sillas.
- `sillas_espera`: Contador de clientes esperando en la sala.
- `barbero_durmiendo`: Bandera lógica del estado del barbero.
- **Lock Global (`mutex`):** Garantiza exclusión mutua para acceder y modificar las variables compartidas de forma atómica.

--- 

### 2. Invariantes del Sistema (Propiedades de Seguridad y Vivacidad)

Según la teoría de sistemas concurrentes, un **invariante** es una condición lógica que siempre debe ser verdadera en cualquier estado de ejecución del sistema.

#### 2.1 Propiedades de Seguridad (Safety)
Garantizan que "nada malo ocurra" durante la ejecución del programa:

- **Invariante 1 (Capacidad de la Sala):**
  $$0 \le sillas\_espera \le CAPACIDAD\_MAX$$
  *Significado:* Jamás puede haber un número negativo de clientes ni tampoco más clientes esperando que las sillas físicamente disponibles.

- **Invariante 2 (Ausencia de Negligencia del Barbero):**
  $$\text{Si } barbero\_durmiendo = True \implies sillas\_espera = 0$$
  *Significado:* El barbero no puede estar durmiendo si hay clientes esperando en la sala.

- **Invariante 3 (Ausencia de Espera Inútil):**
  $$\text{Si } sillas\_espera > 0 \implies barbero\_durmiendo = False$$
  *Significado:* Si hay clientes sentados, el barbero obligatoriamente debe estar despierto (atendiendo o preparándose para atender).

#### 2.2 Propiedades de Vivacidad (Liveness)
Garantizan que "algo bueno ocurra eventualmente":

- **Invariante de Conservación de Clientes:**
  $$Clientes\_Creados = Clientes\_Atendidos + Clientes\_Rechazados$$
  *Significado:* Ningún cliente puede "desaparecer" del sistema; todos deben terminar con éxito o ser rechazados de manera segura.
- **Ausencia de Deadlock:** El barbero y los clientes no deben bloquearse mutuamente de forma permanente si hay capacidad de atención.
- **Ausencia de Starvation:** Todo cliente que logra sentarse en la sala de espera debe ser atendido eventualmente (garantía de orden FIFO).

### 3. Implementación del Validador de Invariantes

Implementamos una clase auditora que realiza aserciones ejecutables (`assert`) sobre el estado de la barbería en tiempo real. Esta clase será llamada en cada paso crítico de sincronización en el Frente 3.

In [22]:
# ===========================================================================
# FRENTE 2 — CLASE DE VALIDACIÓN DE INVARIANTES (Aserciones Ejecutables)
# ===========================================================================

class ValidadorInvariantes:
    def __init__(self, capacidad_max):
        self.capacidad_max = capacidad_max
        self.lock = threading.Lock()  # Lock de auditoría para proteger contadores
        
        # Variables de auditoría para verificar la vivacidad al final de la corrida
        self.total_clientes_creados = 0
        self.total_clientes_atendidos = 0
        self.total_clientes_rechazados = 0

    def registrar_llegada(self):
        """Registra la creación de un nuevo hilo cliente."""
        with self.lock:
            self.total_clientes_creados += 1

    def registrar_atencion(self):
        """Registra que un cliente recibió su corte de cabello exitosamente."""
        with self.lock:
            self.total_clientes_atendidos += 1

    def registrar_rechazo(self):
        """Registra que un cliente se retiró porque la sala de espera estaba llena."""
        with self.lock:
            self.total_clientes_rechazados += 1

    def verificar_invariantes(self, sillas_espera, barbero_durmiendo):
        """
        Evalúa las aserciones de seguridad en el estado actual.
        Si alguna aserción falla, levantará un AssertionError deteniendo el sistema.
        """
        with self.lock:
            # -----------------------------------------------------------------
            # INVARIANTE 1: La cantidad de clientes sentados no excede la capacidad
            # 0 <= sillas_espera <= CAPACIDAD_MAX
            # -----------------------------------------------------------------
            assert 0 <= sillas_espera <= self.capacidad_max, \
                f"Violación Invariante 1: sillas_espera={sillas_espera} fuera de límites [0, {self.capacidad_max}]."
            
            # -----------------------------------------------------------------
            # INVARIANTE 2: Si el barbero duerme, la sala de espera debe estar vacía
            # barbero_durmiendo == True => sillas_espera == 0
            # -----------------------------------------------------------------
            if barbero_durmiendo:
                assert sillas_espera == 0, \
                    f"Violación Invariante 2: El barbero duerme, pero hay {sillas_espera} cliente(s) esperando."
            
            # -----------------------------------------------------------------
            # INVARIANTE 3: Si hay clientes esperando, el barbero no puede estar durmiendo
            # sillas_espera > 0 => barbero_durmiendo == False
            # -----------------------------------------------------------------
            if sillas_espera > 0:
                assert not barbero_durmiendo, \
                    "Violación Invariante 3: Hay clientes esperando, pero el barbero sigue durmiendo."

    def verificar_conservacion(self):
        """
        Verifica al final de la simulación que no se perdieron clientes.
        total_creados == total_atendidos + total_rechazados
        """
        with self.lock:
            total_procesados = self.total_atendidos + self.total_rechazados
            assert self.total_clientes_creados == total_procesados, \
                f"Violación Vivacidad/Conservación: Clientes creados ({self.total_clientes_creados}) " \
                f"no coincide con atendidos+rechazados ({total_procesados})."
            print(f"[Auditoría] Invariante de Conservación verificado con éxito: "
                  f"Creados={self.total_clientes_creados}, Atendidos={self.total_atendidos}, Rechazados={self.total_clientes_rechazados}.")

---

## Frente 3 — Implementación Correcta y Robusta del Peluquero Dormido

### 1. Justificación del Mecanismo de Sincronización (Monitor)

Para resolver de manera definitiva las condiciones de carrera del Frente 1, utilizaremos la abstracción de **Monitor** (Lock + Variables de Condición) mediante el módulo `threading` estándar de Python.

#### 1.1 ¿Qué problemas resuelve este diseño?
- **Exclusión Mutua:** Un `threading.Lock` central garantiza que solo un hilo (ya sea el barbero o cualquier cliente) pueda consultar o alterar las variables de estado (`cola_espera`, `barbero_durmiendo`, etc.) a la vez, eliminando las race conditions.
- **Sincronización Lógica:** Las variables de condición (`threading.Condition`) asociadas al lock permiten que un hilo libere el lock y se bloquee atómicamente hasta recibir una notificación (`wait()`), resolviendo de forma nativa el problema del *Lost Wakeup* (pues la verificación de estado y el bloqueo ocurren de manera indivisible bajo el mismo lock).

#### 1.2 Justificación de Primitivas frente a otras alternativas
- **Locks Puros:** Son insuficientes por sí mismos, ya que el barbero necesitaría hacer *busy waiting* (espera activa) consultando la sala, lo cual desperdicia CPU (OSTEP Cap. 28).
- **Semáforos Clásicos:** Aunque resuelven el problema (como lo detalla Downey en *The Little Book of Semaphores*), son más propensos a errores de programación sutiles debido a su estado interno entero que "recuerda" señales pasadas. Las Variables de Condición acopladas a un Lock expresan de manera explícita la condición lógica de espera.

--- 

### 2. Manejo de Despertares Espurios (`while` vs `if` según OSTEP)

Como se define en **OSTEP Capítulo 30 (p. 7)**, al suspender un hilo con `wait()`, siempre debemos utilizar un bucle `while` para comprobar la condición lógica, y nunca una sentencia `if`:

```python
while condicion_no_cumplida:
    condicion.wait()
```

#### ¿Por qué es obligatorio?
1. **Despertares Espurios (Spurious Wakeups):** El sistema operativo puede despertar a un hilo bloqueado sin que se haya emitido una señal real de forma explícita.
2. **Intercalación de Terceros (Mesa-style semantics):** Cuando un hilo emite `notify()`, el hilo despertado no se ejecuta inmediatamente; pasa a la cola de listos (`READY`). En el intervalo entre que despierta y vuelve a adquirir el Lock, otro hilo cliente podría llegar y sentarse o el barbero ocuparse. Si usamos `if`, el hilo asumiría ciegamente que la condición es válida, lo cual provocaría inconsistencias de estado.

In [23]:
# ===========================================================================
# FRENTE 3 — IMPLEMENTACIÓN CORRECTA (MONITOR DE LA BARBERÍA)
# ===========================================================================

class BarberiaMonitor:
    def __init__(self, capacidad_sillas, validador):
        self.capacidad_sillas = capacidad_sillas
        self.validador = validador
        
        # Lock principal del monitor para exclusión mutua
        self.lock = threading.Lock()
        
        # Variables de condición asociadas al lock central
        self.cond_barbero = threading.Condition(self.lock)   # Espera del barbero
        self.cond_clientes = threading.Condition(self.lock)  # Espera de clientes en sala
        self.cond_corte = threading.Condition(self.lock)     # Espera durante el corte
        
        # Variables de estado internas
        self.cola_espera = []            # Fila FIFO de clientes esperando
        self.barbero_durmiendo = False
        self.cliente_en_silla = None     # ID del cliente en la silla del barbero
        self.corte_finalizado = False    # Estado del corte actual
        self.activo = True               # Flag para finalización ordenada de hilos

    def barbero_esperar_cliente(self):
        """
        Método llamado por el barbero para obtener el siguiente cliente.
        Si la sala está vacía, se duerme en cond_barbero.
        """
        with self.lock:
            # Bucle while (OSTEP Cap 30) para protegerse de despertares espurios
            while len(self.cola_espera) == 0 and self.activo:
                self.barbero_durmiendo = True
                self.validador.verificar_invariantes(len(self.cola_espera), self.barbero_durmiendo)
                self.cond_barbero.wait()  # Libera lock y duerme atómicamente
            
            # Si se ordenó apagar el monitor y no hay clientes, el barbero se retira
            if not self.activo and len(self.cola_espera) == 0:
                return None
                
            self.barbero_durmiendo = False
            
            # Toma el primer cliente de la fila (FIFO)
            cliente_id = self.cola_espera[0]
            
            # Notifica a los clientes en espera para que el correspondiente pase a la silla
            self.cond_clientes.notify_all()
            
            # Espera a que el cliente seleccionado se siente en la silla de corte
            while self.cliente_en_silla != cliente_id:
                self.cond_barbero.wait()
                
            self.corte_finalizado = False
            self.validador.verificar_invariantes(len(self.cola_espera), self.barbero_durmiendo)
            return cliente_id

    def barbero_terminar_corte(self, cliente_id):
        """
        Método llamado por el barbero para finalizar el corte de cabello.
        """
        with self.lock:
            self.corte_finalizado = True
            # Despierta a los clientes en cond_corte (el de la silla reanudará)
            self.cond_corte.notify_all()
            # Libera la silla del barbero
            self.cliente_en_silla = None
            self.validador.verificar_invariantes(len(self.cola_espera), self.barbero_durmiendo)

    def cliente_llegar(self, cliente_id):
        """
        Método llamado por un cliente al entrar a la barbería.
        Retorna True si fue atendido, False si la sala estaba llena.
        """
        with self.lock:
            self.validador.registrar_llegada()
            self.validador.verificar_invariantes(len(self.cola_espera), self.barbero_durmiendo)
            
            # Si la sala de espera está llena, se rechaza de inmediato
            if len(self.cola_espera) >= self.capacidad_sillas:
                self.validador.registrar_rechazo()
                return False
            
            # Toma una silla de espera
            self.cola_espera.append(cliente_id)
            
            # Si el barbero está dormido, lo despierta inmediatamente
            if self.barbero_durmiendo:
                self.cond_barbero.notify_all()
                
            # Bucle while para esperar turno y silla del barbero libre
            while self.cola_espera[0] != cliente_id or self.cliente_en_silla is not None:
                self.cond_clientes.wait()
            
            # Toma el turno: sale de la cola y se sienta en la silla de corte
            self.cola_espera.remove(cliente_id)
            self.cliente_en_silla = cliente_id
            
            # Notifica al barbero que ya está sentado listo para iniciar el corte
            self.cond_barbero.notify_all()
            
            # Espera en la silla hasta que el barbero finalice el corte
            while not self.corte_finalizado or self.cliente_en_silla != cliente_id:
                self.cond_corte.wait()
                
            self.validador.registrar_atencion()
            self.validador.verificar_invariantes(len(self.cola_espera), self.barbero_durmiendo)
            return True

    def detener(self):
        """
        Detiene el monitor ordenadamente al finalizar las pruebas.
        """
        with self.lock:
            self.activo = False
            self.cond_barbero.notify_all()
            self.cond_clientes.notify_all()

### 3. Hilos de Ejecución (Barbero y Clientes)

Los hilos invocan los métodos del monitor. El corte de cabello (`time.sleep`) se simula **fuera del monitor** para asegurar que el lock no se retenga durante operaciones prolongadas.

In [24]:
# ===========================================================================
# FRENTE 3 — FUNCIONES DE HILOS (EJECUCIÓN FUERA DEL LOCK)
# ===========================================================================

def barbero_thread_correct(monitor):
    """Hilo del barbero: atiende clientes continuamente."""
    while True:
        # 1. Espera y selecciona un cliente bajo exclusión mutua atómica del Monitor
        cliente_id = monitor.barbero_esperar_cliente()
        
        # Si retorna None es señal de que las pruebas terminaron y la barbería cierra
        if cliente_id is None:
            break
            
        # 2. Simulación del corte de cabello FUERA del monitor
        # Esto es vital para permitir concurrencia de llegadas en las sillas de espera.
        duracion_corte = random.uniform(0.02, 0.06)
        time.sleep(duracion_corte)
        
        # 3. Finaliza el corte y despide al cliente bajo exclusión mutua del Monitor
        monitor.barbero_terminar_corte(cliente_id)

def cliente_thread_correct(monitor, cliente_id, retardo_llegada):
    """Hilo del cliente: viaja, llega a la barbería y solicita atención."""
    # Simula el tiempo de viaje a la barbería
    time.sleep(retardo_llegada)
    
    # Intenta ingresar y ser atendido
    fue_atendido = monitor.cliente_llegar(cliente_id)

---

## Frente 4 — Harness de Pruebas Adversariales

### 1. Diseño de Escenarios

Para garantizar que nuestra implementación correcta funciona bajo cualquier carga e intercalación, diseñamos un sistema de pruebas que inyecta cargas de trabajo adversas y verifica el cumplimiento estricto de todos los invariantes definidos en el Frente 2.

#### 1.1 Escenarios de Estrés Evaluados
1. **Llegada en Ráfaga Simultánea (Burst):** Muchos clientes (10) llegan en el mismo instante de tiempo ($T=0.0$). La capacidad es reducida (3 sillas). Se debe validar que la sala nunca se desborde, exactamente 3 se sienten a esperar, el resto (7) sean rechazados, y que nadie quede en un *lost wakeup*.
2. **Llegadas Secuenciales Lentas:** Clientes que llegan secuencialmente espaciados. El barbero se duerme entre clientes, y es despertado sucesivamente sin condiciones de carrera.
3. **Caso Límite (Sala de Espera sin Sillas):** Capacidad de sillas = 0. En este caso extremo, un cliente solo puede ser atendido si encuentra al barbero durmiendo directamente. Cualquier cliente que llegue mientras el barbero esté ocupado debe ser rechazado inmediatamente.

In [25]:
# ===========================================================================
# FRENTE 4 — HARNESS DE PRUEBAS ADVERSARIALES Y AUDITORÍA DE INVARIANTES
# ===========================================================================

def run_test_scenario(nombre_prueba, capacidad_sillas, delay_clientes):
    """
    Ejecuta un escenario de prueba adversarial con parámetros controlados,
    ejecutando en tiempo real las aserciones de invariantes de seguridad y vivacidad.
    """
    print("=" * 75)
    print(f"INICIANDO ESCENARIO: {nombre_prueba.upper()}")
    print(f"Configuración: {capacidad_sillas} sillas | Hilos clientes: {len(delay_clientes)}")
    print("=" * 75)
    
    # Instanciar el validador y el monitor
    validador = ValidadorInvariantes(capacidad_sillas)
    monitor = BarberiaMonitor(capacidad_sillas, validador)
    
    # Crear e iniciar el hilo del barbero
    t_barbero = threading.Thread(target=barbero_thread_correct, args=(monitor,), name="Barbero")
    t_barbero.start()
    
    # Crear e iniciar los hilos de clientes
    hilos_clientes = []
    for idx, delay in enumerate(delay_clientes):
        validador.registrar_llegada()
        t_cli = threading.Thread(
            target=cliente_thread_correct, 
            args=(monitor, idx + 1, delay),
            name=f"Cliente-{idx + 1}"
        )
        hilos_clientes.append(t_cli)
        t_cli.start()
        
    # Esperar a que todos los clientes terminen sus solicitudes de servicio
    for t_cli in hilos_clientes:
        t_cli.join()
        
    # Apagar el barbero de forma segura una vez procesados todos los clientes
    monitor.detener()
    t_barbero.join()
    
    # Ejecutar la verificación final del invariante de conservación de hilos
    validador.verificar_conservacion()
    print("✅ Prueba finalizada exitosamente. Todos los invariantes se cumplieron.\n")